# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-14: HR RhythmMamba Optuna Hyperparameter Search (resumable)

### What this notebook does
The RhythmMamba twin of NB_P3_13 - a pruned, resumable Optuna search over RhythmMamba (screen winner on raw MAE: 4.92 M params, val HR MAE 6.83). Same loader / loss / eval / resumable-study machinery; only the model, its fp32 forward, and a few budget knobs differ. Every trial records its per-dataset MAE / bias / r as user attributes for the later comparison. 


### RhythmMamba-specific cost + risks (read first)
- **Slower to tune than PhysNet.** RhythmMamba only 'clicks' around epoch ~13, so the proxy runs `PROXY_EPOCHS=16` and the pruner is held off until epoch 13 so it does not kill late-clicking configs. Expect ~45-55 min/trial -> ~10 trials per commit, so budget more commits.
- **mamba-ssm / causal-conv1d** compile CUDA kernels in Section 1 (several minutes each commit). If the import fails on the current Kaggle image, RhythmMamba can't run here - fall back to tuning PhysNet only.
- Search space omits dropout (RhythmMamba does not expose one); it tunes lr / weight_decay / lambda_freq / aug_prob / batch_size. No frames/faces are rendered.

## 1. Imports & Config (+ clone RhythmMamba, install mamba-ssm)

In [ ]:
import os, sys, time, random, glob, gc, shutil, importlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

try:
    import optuna
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
    import optuna

# Clone RhythmMamba + install the Mamba CUDA deps (may take several minutes).
if not os.path.exists('RhythmMamba'):
    os.system('git clone --depth 1 https://github.com/zizheng-guo/RhythmMamba')
sys.path.append('RhythmMamba')
for d in glob.glob('RhythmMamba/**/', recursive=True):
    sys.path.append(d)
os.system('pip install einops -q')
os.system('pip install causal-conv1d --no-build-isolation -q')
os.system('pip install mamba-ssm --no-build-isolation -q')
try:
    import mamba_ssm
    print('mamba-ssm OK:', getattr(mamba_ssm, '__version__', 'imported'))
except Exception as e:
    print('WARNING mamba-ssm import failed:', repr(e), '-> RhythmMamba cannot run on this image.')

DATA_DIR = Path('/kaggle/input/datasets/cezarytubacki/phase3-hr-train-dataset')
H5_FILES = sorted([f.name for f in DATA_DIR.glob('*.h5')])

RUN_RETRAIN = False

CLIP_LEN = 160; IMG_SIZE = 72; NUM_WORKERS = 2; VAL_FRAC = 0.2; SEED = 42
SPEED_MIN, SPEED_MAX = 0.7, 2.0; SQI_FLOOR = 0.05; HR_LOW, HR_HIGH = 0.66, 3.0
CLIPS_PER_REC_SEARCH = 1; CLIPS_PER_REC_FINAL = 2

PROXY_EPOCHS = 16 # past RhythmMamba's ~epoch-13 'click'
WARMUP_STEPS = 13 # pruner is held off until here (protect late-clicking configs)
FINAL_EPOCHS = 30
N_TRIALS = 24
TIMEOUT_H = 9.5

DATA_VERSION = 'v1_mcd_ubfc_dlcn'
DB_NAME = 'rhythmmamba_optuna.db'
WORK_DB = f'/kaggle/working/{DB_NAME}'
STORAGE = f'sqlite:///{WORK_DB}'
STUDY_NAME = f'rhythmmamba_{DATA_VERSION}'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True
print('device:', device, '| DATA_DIR:', DATA_DIR, '| files:', H5_FILES)
print('mode:', 'RETRAIN best' if RUN_RETRAIN else 'SEARCH only', '| study:', STUDY_NAME)

## 2. Data Loader — index, subject-wise split, dataset (parameterized)

In [ ]:
def build_index(data_dir, h5_files, sqi_floor=SQI_FLOOR, clip_len=CLIP_LEN):
    index = []
    for fn in h5_files:
        path = Path(data_dir) / fn
        if not path.exists():
            print('missing', path); continue
        with h5py.File(path, 'r') as f:
            for name in f.keys():
                a = f[name].attrs
                if not bool(a.get('usable', True)):
                    continue
                if float(a.get('cardiac_sqi', 0.0)) < sqi_floor:
                    continue
                n = int(a['n_frames'])
                if n < clip_len:
                    continue
                index.append(dict(file=fn, group=name, dataset=str(a.get('dataset', fn)),
                                  subject=str(a.get('subject_id', name)), n_frames=n,
                                  fps=float(a.get('fps', 30.0)), sqi=float(a.get('cardiac_sqi', 0.0))))
    return index


def subject_split(index, val_frac=VAL_FRAC, seed=SEED):
    subjects = sorted({r['dataset'] + '/' + r['subject'] for r in index})
    rng = random.Random(seed); rng.shuffle(subjects)
    n_val = max(1, int(round(len(subjects) * val_frac)))
    val_subj = set(subjects[:n_val])
    tr = [r for r in index if (r['dataset'] + '/' + r['subject']) not in val_subj]
    va = [r for r in index if (r['dataset'] + '/' + r['subject']) in val_subj]
    return tr, va, val_subj


def compute_hr(bvp, fps, low=HR_LOW, high=HR_HIGH):
    x = np.asarray(bvp, dtype=np.float64); x = x - x.mean()
    if x.size < 16 or not np.all(np.isfinite(x)) or np.std(x) < 1e-9:
        return np.nan
    p = np.abs(np.fft.rfft(x * np.hanning(x.size))) ** 2
    fr = np.fft.rfftfreq(x.size, 1.0 / fps)
    b = (fr >= low) & (fr <= high)
    if not b.any() or p[b].sum() <= 0:
        return np.nan
    return float(fr[b][int(np.argmax(p[b]))] * 60.0)


class RPPGClipDataset(Dataset):
    def __init__(self, index, data_dir, clip_len=CLIP_LEN, augment=False, clips_per_rec=1,
                 aug_prob=0.5, speed_range=(SPEED_MIN, SPEED_MAX), seed=SEED):
        self.index = index; self.data_dir = Path(data_dir); self.clip_len = clip_len
        self.augment = augment; self.clips_per_rec = clips_per_rec
        self.aug_prob = aug_prob; self.speed_range = speed_range
        self._files = {}; self._rng = np.random.default_rng(seed)

    def _h5(self, fn):
        h = self._files.get(fn)
        if h is None:
            h = h5py.File(self.data_dir / fn, 'r'); self._files[fn] = h
        return h

    def __len__(self):
        return len(self.index) * self.clips_per_rec

    def __getitem__(self, i):
        rec = self.index[i % len(self.index)]; g = self._h5(rec['file'])[rec['group']]
        n, T = rec['n_frames'], self.clip_len
        if self.augment and self._rng.random() < self.aug_prob:
            src_len = int(round(T * float(self._rng.uniform(*self.speed_range))))
            src_len = max(T, min(src_len, n))
        else:
            src_len = T
        max_start = n - src_len
        start = int(self._rng.integers(0, max_start + 1)) if (self.augment and max_start > 0) else max_start // 2
        frames_src = g['frames'][start:start + src_len].astype(np.float32)
        bvp_src = g['bvp'][start:start + src_len].astype(np.float32)
        pos = np.linspace(0, src_len - 1, T)
        frames = frames_src[np.rint(pos).astype(int)]
        bvp = np.interp(pos, np.arange(src_len), bvp_src)
        hr_bpm = compute_hr(bvp, rec['fps'])
        if self.augment and self._rng.random() < 0.5:
            frames = frames[:, :, ::-1, :]
        m = frames.mean(axis=(0, 1, 2), keepdims=True); sd = frames.std(axis=(0, 1, 2), keepdims=True) + 1e-6
        frames = np.ascontiguousarray((frames - m) / sd)
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2).float()
        bvp = (bvp - bvp.mean()) / (bvp.std() + 1e-6)
        bvp = torch.from_numpy(bvp.astype(np.float32))
        return dict(frames=frames, bvp=bvp,
                    hr=torch.tensor(hr_bpm if np.isfinite(hr_bpm) else 0.0, dtype=torch.float32),
                    sqi=torch.tensor(rec['sqi'], dtype=torch.float32),
                    fps=torch.tensor(rec['fps'], dtype=torch.float32),
                    dataset=rec['dataset'])


def worker_init_fn(worker_id):
    info = torch.utils.data.get_worker_info(); ds = info.dataset
    ds._files = {}; ds._rng = np.random.default_rng(SEED + 1000 * (worker_id + 1) + int(info.seed % 100000))


def make_loader(index, augment, batch_size=8, aug_prob=0.5, clips_per_rec=1, persistent=False):
    ds = RPPGClipDataset(index, DATA_DIR, augment=augment, clips_per_rec=clips_per_rec, aug_prob=aug_prob)
    return DataLoader(ds, batch_size=batch_size, shuffle=augment, num_workers=NUM_WORKERS,
                      worker_init_fn=worker_init_fn, pin_memory=True, drop_last=augment,
                      persistent_workers=(persistent and NUM_WORKERS > 0))

## 3. Model — RhythmMamba (official repo, fp32 forward)

In [ ]:
RhythmMambaClass = None
for modname in ['neural_methods.model.RhythmMamba', 'model.RhythmMamba', 'RhythmMamba']:
    try:
        m = importlib.import_module(modname)
        RhythmMambaClass = getattr(m, 'RhythmMamba')
        print('imported RhythmMamba from', modname)
        break
    except Exception as e:
        print('import', modname, 'failed:', repr(e))
if RhythmMambaClass is None:
    print('RhythmMamba class NOT found. Repo model .py files:')
    for pyf in glob.glob('RhythmMamba/**/*.py', recursive=True):
        print('   ', pyf)
    raise RuntimeError('Could not import RhythmMamba - paste this listing.')


def build_rhythmmamba():
    last = None
    for kw in [dict(), dict(frames=CLIP_LEN), dict(dim=96)]:
        try:
            return RhythmMambaClass(**kw)
        except Exception as e:
            last = e
    raise last


def rm_forward(model, x):
    '''Our clips are [B,C,T,H,W]; RhythmMamba expects [B,T,C,H,W]. Forced fp32 (its internal FFT cannot run in fp16 for non-power-of-2 sizes). Returns [B,T].'''
    xx = x.permute(0, 2, 1, 3, 4).contiguous().float()
    with torch.autocast('cuda', enabled=False):
        out = model(xx)
    if isinstance(out, (tuple, list)):
        out = out[0]
    if out.dim() == 3 and out.shape[-1] == 1:
        out = out.squeeze(-1)
    if out.dim() == 3 and out.shape[1] == 1:
        out = out.squeeze(1)
    return out

## 4. Losses & Evaluation Functions

In [ ]:
def neg_pearson(pred, target):
    pred = pred - pred.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)
    num = (pred * target).sum(dim=1)
    den = torch.sqrt((pred ** 2).sum(dim=1) * (target ** 2).sum(dim=1) + 1e-8)
    return 1.0 - num / (den + 1e-8)


def freq_loss(pred, target, fps=30.0, low=HR_LOW, high=HR_HIGH):
    pred = pred.float(); target = target.float()
    pred = pred - pred.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)
    Pp = torch.abs(torch.fft.rfft(pred, dim=1)) ** 2
    Pt = torch.abs(torch.fft.rfft(target, dim=1)) ** 2
    freqs = torch.fft.rfftfreq(pred.shape[1], 1.0 / fps).to(pred.device)
    band = (freqs >= low) & (freqs <= high)
    Pp = Pp[:, band]; Pt = Pt[:, band]
    Pp = Pp / (Pp.sum(dim=1, keepdim=True) + 1e-8)
    Pt = Pt / (Pt.sum(dim=1, keepdim=True) + 1e-8)
    return ((Pp - Pt) ** 2).sum(dim=1)


def hr_from_bvp(bvp, fps, low=HR_LOW, high=HR_HIGH):
    x = np.asarray(bvp, dtype=np.float64); x = x - x.mean()
    if x.std() < 1e-8:
        return np.nan
    p = np.abs(np.fft.rfft(x * np.hanning(len(x)))) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / fps)
    b = (f >= low) & (f <= high)
    if not b.any() or p[b].sum() <= 0:
        return np.nan
    return float(f[b][int(np.argmax(p[b]))] * 60.0)


def train_one_epoch(model, loader, opt, scaler, lam):
    model.train()
    for batch in loader:
        frames = batch['frames'].to(device, non_blocking=True)
        bvp = batch['bvp'].to(device, non_blocking=True)
        sqi = batch['sqi'].to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        pred = rm_forward(model, frames).float()
        per = neg_pearson(pred, bvp) + lam * freq_loss(pred, bvp)
        w = sqi.clamp(min=1e-3)
        loss = (per * w).sum() / w.sum()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()


@torch.no_grad()
def evaluate(model, loader, tag='val', verbose=True, return_detail=False):
    '''Returns overall HR MAE. With return_detail=True also returns a dict of per-dataset MAE / bias / rmse / r / n.'''
    model.eval(); rows = []
    for batch in loader:
        frames = batch['frames'].to(device, non_blocking=True)
        pred = rm_forward(model, frames).float().cpu().numpy()
        hr_true = batch['hr'].numpy(); fps = batch['fps'].numpy(); ds = batch['dataset']
        for i in range(len(pred)):
            if not (40.0 <= hr_true[i] <= 180.0):
                continue
            hp = hr_from_bvp(pred[i], float(fps[i]))
            if np.isfinite(hp):
                rows.append((ds[i], float(hr_true[i]), hp))
    df = pd.DataFrame(rows, columns=['dataset', 'hr_true', 'hr_pred'])
    if len(df) == 0:
        return (1e9, {}) if return_detail else 1e9
    df['ae'] = (df.hr_pred - df.hr_true).abs()
    mae = float(df.ae.mean())
    if verbose:
        rmse = float(np.sqrt(((df.hr_pred - df.hr_true) ** 2).mean()))
        r = float(np.corrcoef(df.hr_true, df.hr_pred)[0, 1]) if len(df) > 2 else float('nan')
        print(f'  [{tag}] MAE {mae:.2f} | RMSE {rmse:.2f} | r {r:.3f} | n {len(df)}',
              '| per-dataset', df.groupby('dataset')['ae'].mean().round(2).to_dict())
    if return_detail:
        d = (df.hr_pred - df.hr_true)
        detail = {'per_dataset': df.groupby('dataset')['ae'].mean().round(3).to_dict(),
                  'bias': round(float(d.mean()), 3),
                  'rmse': round(float(np.sqrt((d ** 2).mean())), 3),
                  'r': (round(float(np.corrcoef(df.hr_true, df.hr_pred)[0, 1]), 3) if len(df) > 2 else None),
                  'n': int(len(df))}
        return mae, detail
    return mae

## 5. Optuna Objective (proxy training + late-safe pruning)
Every trial records its best-epoch per-dataset breakdown as user attributes (`per_dataset`, `bias`, `rmse`, `r`, `n`, `best_epoch`) for the later comparison notebook.

In [ ]:
index = build_index(DATA_DIR, H5_FILES)
print('usable recordings:', len(index), '| by dataset:', dict(Counter(r['dataset'] for r in index)))
train_idx, val_idx, val_subj = subject_split(index)
print('train recs:', len(train_idx), '| val recs:', len(val_idx), '| val subjects:', len(val_subj))

VAL_LOADER = make_loader(val_idx, augment=False, batch_size=16, clips_per_rec=1, persistent=False)

_m = build_rhythmmamba().to(device)
print('RhythmMamba params:', round(sum(p.numel() for p in _m.parameters()) / 1e6, 2), 'M | baseline to beat: 6.83')
with torch.no_grad():
    _o = rm_forward(_m, next(iter(VAL_LOADER))['frames'][:2].to(device))
print('forward sanity:', tuple(_o.shape), '(expect [2,', CLIP_LEN, '])')
del _m, _o; gc.collect(); torch.cuda.empty_cache()


def objective(trial):
    lr = trial.suggest_float('lr', 3e-4, 3e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    lam = trial.suggest_float('lambda_freq', 0.05, 0.5)
    aug_prob = trial.suggest_float('aug_prob', 0.3, 0.7)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])

    torch.manual_seed(SEED)
    train_loader = make_loader(train_idx, augment=True, batch_size=batch_size,
                               aug_prob=aug_prob, clips_per_rec=CLIPS_PER_REC_SEARCH, persistent=False)
    model = build_rhythmmamba().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PROXY_EPOCHS)
    scaler = GradScaler('cuda')
    best = 1e9
    try:
        for epoch in range(PROXY_EPOCHS):
            train_one_epoch(model, train_loader, opt, scaler, lam)
            sched.step()
            mae, detail = evaluate(model, VAL_LOADER, verbose=False, return_detail=True)
            if mae < best:
                best = mae
                trial.set_user_attr('per_dataset', detail.get('per_dataset', {}))
                trial.set_user_attr('bias', detail.get('bias'))
                trial.set_user_attr('rmse', detail.get('rmse'))
                trial.set_user_attr('r', detail.get('r'))
                trial.set_user_attr('n', detail.get('n'))
                trial.set_user_attr('best_epoch', epoch)
            trial.report(mae, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            torch.cuda.empty_cache(); raise optuna.TrialPruned()
        raise
    finally:
        del model, opt, train_loader
        gc.collect(); torch.cuda.empty_cache()
    return best

## 6. Run the Search (resumable SQLite study)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

if not Path(WORK_DB).exists():
    prev = next((p for p in Path('/kaggle/input').rglob(DB_NAME)), None)
    if prev is not None:
        shutil.copy(prev, WORK_DB); print('resumed prior study db from', prev)
    else:
        print('no prior db in /kaggle/input -> starting a fresh study')

sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)
pruner = optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=WARMUP_STEPS, interval_steps=1)
study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner,
                            study_name=STUDY_NAME, storage=STORAGE, load_if_exists=True)

# Warm-start ONLY a brand-new study with the screen's known-good RhythmMamba config (6.83).
if len([t for t in study.trials if t.state.name in ('COMPLETE', 'PRUNED')]) == 0:
    study.enqueue_trial({'lr': 1e-3, 'weight_decay': 1e-4, 'lambda_freq': 0.20, 'aug_prob': 0.5, 'batch_size': 8})
    print('fresh study -> enqueued the screen config as warm-start')
print(f'study "{STUDY_NAME}" | trials so far: {len(study.trials)}')


def _log_cb(study, trial):
    val = None if trial.value is None else round(trial.value, 3)
    pd_ = trial.user_attrs.get('per_dataset')
    print(f'trial {trial.number:3d} | {trial.state.name:9s} | MAE {val} | best {round(study.best_value, 3)} | per-ds {pd_}')

if not RUN_RETRAIN:
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS, timeout=int(TIMEOUT_H * 3600),
                   callbacks=[_log_cb], gc_after_trial=True)
    print(f'\nsearch stopped after {(time.time() - t0) / 3600:.2f} h | total trials: {len(study.trials)}')
    print('BEST val HR MAE:', round(study.best_value, 3), '(screen RhythmMamba was 6.83)')
    print('BEST params:', {k: (round(v, 5) if isinstance(v, float) else v) for k, v in study.best_params.items()})
    print('BEST per-dataset:', study.best_trial.user_attrs.get('per_dataset'))
else:
    print('RUN_RETRAIN=True -> skipping search; using existing study best for Section 8.')

## 7. Inspect Results

In [ ]:
print('trial states:', dict(Counter(t.state.name for t in study.trials)))
try:
    imp = optuna.importance.get_param_importances(study)
    print('\nparam importances:')
    for k, v in imp.items():
        print(f'  {k:14s} {v:.3f}')
except Exception as e:
    print('importance needs >=2 completed trials:', e)

rows = []
for t in study.trials:
    if t.state.name != 'COMPLETE':
        continue
    rows.append({'trial': t.number, 'MAE': round(t.value, 3), 'per_dataset': t.user_attrs.get('per_dataset'),
                 'bias': t.user_attrs.get('bias'), 'r': t.user_attrs.get('r'), **t.params})
top = pd.DataFrame(rows).sort_values('MAE')
print('\ntop 10 completed trials (with per-dataset):')
print(top.head(10).to_string(index=False))

## 8. Retrain the Best Config (full 30 epochs) + Honest Eval
Runs only when `RUN_RETRAIN=True`.

In [ ]:
if not RUN_RETRAIN:
    print('RUN_RETRAIN is False -> skipping retrain. Set it True in Section 1 for the final run.')
else:
    bp = study.best_params
    print('retraining RhythmMamba with best config:', bp)
    torch.manual_seed(SEED)
    tr_loader = make_loader(train_idx, augment=True, batch_size=bp['batch_size'],
                            aug_prob=bp['aug_prob'], clips_per_rec=CLIPS_PER_REC_FINAL, persistent=True)
    va_loader = make_loader(val_idx, augment=False, batch_size=16, clips_per_rec=1, persistent=True)
    model = build_rhythmmamba().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FINAL_EPOCHS)
    scaler = GradScaler('cuda')
    best = 1e9
    for epoch in range(FINAL_EPOCHS):
        train_one_epoch(model, tr_loader, opt, scaler, bp['lambda_freq'])
        sched.step()
        mae = evaluate(model, va_loader, tag=f'ep{epoch + 1}')
        if mae < best:
            best = mae; torch.save(model.state_dict(), 'rhythmmamba_tuned_best.pt'); print('   * new best', round(best, 2))
    print('TUNED best val HR MAE:', round(best, 2), '(screen RhythmMamba was 6.83)')

In [ ]:
if RUN_RETRAIN and Path('rhythmmamba_tuned_best.pt').exists():
    import matplotlib.pyplot as plt
    model.load_state_dict(torch.load('rhythmmamba_tuned_best.pt', map_location=device)); model.eval()

    @torch.no_grad()
    def collect_predictions(model, loader):
        rows = []
        for batch in loader:
            frames = batch['frames'].to(device, non_blocking=True)
            pred = rm_forward(model, frames).float().cpu().numpy()
            hr_true = batch['hr'].numpy(); fps = batch['fps'].numpy(); ds = batch['dataset']
            for i in range(len(pred)):
                if not (40.0 <= hr_true[i] <= 180.0):
                    continue
                hp = hr_from_bvp(pred[i], float(fps[i]))
                if np.isfinite(hp):
                    rows.append((ds[i], float(hr_true[i]), hp))
        return pd.DataFrame(rows, columns=['dataset', 'hr_true', 'hr_pred'])

    pred_df = collect_predictions(model, va_loader)

    def metrics_block(g):
        d = (g['hr_pred'] - g['hr_true']).values
        sd = d.std(ddof=1) if len(d) > 1 else np.nan
        return pd.Series({'n': len(g), 'MAE': np.abs(d).mean(), 'RMSE': np.sqrt((d ** 2).mean()),
                         'bias': d.mean(), 'SD': sd, 'LoA_low': d.mean() - 1.96 * sd,
                         'LoA_high': d.mean() + 1.96 * sd,
                         'Pearson_r': np.corrcoef(g['hr_true'], g['hr_pred'])[0, 1] if len(g) > 2 else np.nan})

    summary = pred_df.groupby('dataset').apply(metrics_block, include_groups=False)
    overall = metrics_block(pred_df); overall.name = 'ALL'
    print(pd.concat([summary, overall.to_frame().T]).round(2).to_string())

    colors = {'MCD': 'tab:red', 'DLCN': 'tab:blue', 'UBFC-rPPG': 'tab:green'}
    diff = (pred_df.hr_pred - pred_df.hr_true).values
    bias, sd = diff.mean(), diff.std(ddof=1)
    r = np.corrcoef(pred_df.hr_true, pred_df.hr_pred)[0, 1]
    lim = [40, 180]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
    for ds, g in pred_df.groupby('dataset'):
        ax[0].scatter(g.hr_true, g.hr_pred, s=12, alpha=0.5, color=colors.get(ds, 'gray'), label=ds)
    ax[0].plot(lim, lim, 'k--', lw=1, label='identity')
    slope, intc = np.polyfit(pred_df.hr_true, pred_df.hr_pred, 1)
    ax[0].plot(np.array(lim), intc + slope * np.array(lim), 'k-', lw=1.2, label=f'fit (slope {slope:.2f})')
    ax[0].set(xlim=lim, ylim=lim, xlabel='Reference HR (bpm)', ylabel='Predicted HR (bpm)',
              title=f'Predicted vs reference HR (r = {r:.2f})', aspect='equal')
    ax[0].legend(fontsize=8, loc='upper left'); ax[0].grid(alpha=0.3)
    for ds, g in pred_df.groupby('dataset'):
        m = (g.hr_true + g.hr_pred) / 2; d = g.hr_pred - g.hr_true
        ax[1].scatter(m, d, s=12, alpha=0.5, color=colors.get(ds, 'gray'), label=ds)
    ax[1].axhline(bias, color='k', lw=1.2, label=f'bias {bias:.1f}')
    ax[1].axhline(bias + 1.96 * sd, color='k', ls='--', lw=1, label=f'+1.96SD {bias + 1.96 * sd:.1f}')
    ax[1].axhline(bias - 1.96 * sd, color='k', ls='--', lw=1, label=f'-1.96SD {bias - 1.96 * sd:.1f}')
    ax[1].set(xlabel='Mean of predicted & reference HR (bpm)', ylabel='Predicted - reference (bpm)', title='Bland-Altman')
    ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
    ax[2].hist(diff, bins=40, color='tab:gray', alpha=0.85, edgecolor='white')
    ax[2].axvline(0, color='k', lw=1); ax[2].axvline(bias, color='tab:red', lw=1.3, label=f'bias {bias:.1f}')
    ax[2].set(xlabel='Predicted - reference HR (bpm)', ylabel='count', title=f'Error distribution (MAE {np.abs(diff).mean():.2f})')
    ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('(eval runs after a RUN_RETRAIN=True retrain)')